# Experimentação do prompt de Text-to-SQL

O prompt que a API usa está em `server/prompts/`. Este notebook é o lugar para comparar versões antes de trocar o arquivo que vai para produção.

Rode o kernel com o diretório do projeto como raiz, no virtualenv `/home/andreluis/Labs/python/.venv`.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
sys.path.insert(0, str(repo / "server"))

from app.llm.prompts import render
from app.sql_guard.validator import SqlRejected, validate

print((repo / "server" / "prompts" / "text_to_sql_system.md").read_text(encoding="utf-8")[:800])


## O que variar

1. Sem o chunk de métricas, o modelo tende a usar `produtos.preco` no lugar do preço cobrado.
2. Sem os três exemplos do `text_to_sql_user.md`, perguntas vagas viram SQL com filtro inventado.
3. Pedir JSON com `intent` separa esclarecimento de consulta. Sem isso, o modelo devolve SQL e texto misturados.

Anote a versão do prompt no `docs/avaliacao.md` depois de `python -m app.eval.run_eval --mode live`. Não copie um percentual de uma execução antiga.


In [ ]:
amostras = [
    'SELECT COUNT(*) AS valor FROM analista.pedidos',
    'DROP TABLE analista.pedidos',
    'SELECT * FROM analista.usuarios',
]
for sql in amostras:
    try:
        print('ACEITO', validate(sql))
    except SqlRejected as exc:
        print('RECUSADO', exc.reason)
